# BEIR evaluation: FiQA-2018, retrieval only, QAChat's real pipeline

[BEIR](https://github.com/beir-cellar/beir) is a published benchmark suite for retrieval evaluation; this notebook runs its FiQA-2018 task against QAChat's own real retrieval pipeline.

**The Problem.** Given a real financial question (e.g. *"What is considered a business expense on a business trip?"*), can QAChat's real retrieval pipeline find the forum post that actually answers it, out of a corpus of tens of thousands of unrelated posts?

**The Data.** FiQA-2018: 648 test queries with ~2.6 relevant documents each on average, searched against the full official 57,638-document corpus of financial forum posts. No subsampling -- FiQA's corpus is small enough to embed in full locally.

**The Metrics.** NDCG, MAP, Recall, and Precision at several cutoffs, computed via `pytrec_eval` through `evaluate_retrieval` -- BEIR's own scoring. NDCG@10 is BEIR's own primary metric.

**What's different from the original benchmark.** BEIR is retrieval-only by construction (`qrels` are relevance judgments, not reference answer text), so this notebook stays retrieval-only rather than adding a generation layer on top -- QAChat's own engineering surface is chunking, hybrid search, and retrieval planning, not the small local chat model it wires in, and a retrieval-only score isolates the variable QAChat actually controls. A few structural adaptations follow from QAChat's own design. BEIR scores document ids directly, but QAChat's retrieval identity is `Chunk.identifier`, so each FiQA document is written as its own `.txt` file named by its original BEIR id (see section 6), and results are max-aggregated per document id before scoring (section 9) rather than assumed 1:1, since a rare long post can still split into more than one chunk. There's no code channel -- `hybrid_search` runs semantic + lexical only, since `code_search`/`build_code_graph` have nothing to find in prose. Qwen3's embedding model is instruction-tuned, and neither of QAChat's built-in instructions (software repository, academic literature) fits financial forum QA, so this notebook defines its own `FIQA_RETRIEVAL_INSTRUCTION`. No leave-one-out is needed -- unlike code benchmarks, BEIR's query and corpus id spaces are already disjoint by construction. Retrieval oversamples to 100 candidates per query to cover `evaluate_retrieval`'s full k range. FiQA's corpus/query ids are strings while the qrels' ids are int64, so every join below casts explicitly rather than relying on Python's `==` to paper over the mismatch. And because max-aggregation changes the retrieval unit from what BEIR's own published table measures (one embedding per document, not per chunk), the notebook also runs a whole-document baseline that matches that granularity exactly, to give a number directly comparable to the published range.


## 1. Prepare QAChat

Locate the QAChat repo root, put it on the import path, and run QAChat's notebook setup.

In [ ]:
from pathlib import Path

root = Path.cwd()
while not (root / "qachat").is_dir() and root != root.parent:
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Benchmark-only dependencies

`datasets` pulls FiQA from HuggingFace (`BeIR/fiqa` for corpus/queries, `BeIR/fiqa-qrels` for relevance judgments). `pytrec_eval` is already a `benchmarks/common.py` dependency (`evaluate_retrieval`).


In [ ]:
import datasets

print(f"datasets version: {datasets.__version__}")

## 3. Load FiQA queries and the test qrels

Test-split qrels only (648 queries, ~2.6 relevant docs each) -- BEIR
reserves train/dev for tuning a retriever, test for reporting. Queries are
filtered down to the ones that actually appear in the test qrels, and every
id is cast to `str` explicitly -- `BeIR/fiqa`'s `_id` columns are strings,
`BeIR/fiqa-qrels`' `query-id`/`corpus-id` columns are int64, and relying on
Python's cross-type `==` here would silently match nothing.

In [ ]:
from collections import defaultdict

all_queries_rows = datasets.load_dataset("BeIR/fiqa", "queries")["queries"]
qrels_test_rows = datasets.load_dataset("BeIR/fiqa-qrels", split="test")

qrels: dict[str, dict[str, int]] = defaultdict(dict)
for row in qrels_test_rows:
    query_id = str(row["query-id"])
    corpus_id = str(row["corpus-id"])
    qrels[query_id][corpus_id] = int(row["score"])
qrels = dict(qrels)

query_text_by_id = {str(row["_id"]): row["text"] for row in all_queries_rows}

missing_query_ids = set(qrels) - set(query_text_by_id)
if missing_query_ids:
    raise RuntimeError(
        f"{len(missing_query_ids)} qrels query ids have no matching query "
        "text -- id casting or dataset version mismatch."
    )

fiqa_queries = {qid: query_text_by_id[qid] for qid in qrels}

relevant_doc_count = sum(len(docs) for docs in qrels.values())
print(f"Loaded {len(fiqa_queries)} test queries.")
print(
    f"{relevant_doc_count} relevance judgments "
    f"({relevant_doc_count / len(fiqa_queries):.2f} relevant docs/query)."
)

## 4. Load the FiQA corpus

The full official 57,638-document corpus. `CORPUS_SAMPLE_SIZE` below is a smoke-test knob: set it to a small number
for a first pass, `None` for the real run.

In [ ]:
CORPUS_SAMPLE_SIZE = None  # None for the full 57,638-document corpus

corpus_rows = datasets.load_dataset("BeIR/fiqa", "corpus")["corpus"]
if CORPUS_SAMPLE_SIZE is not None:
    corpus_rows = corpus_rows.select(
        range(min(CORPUS_SAMPLE_SIZE, len(corpus_rows)))
    )

fiqa_corpus = {
    str(row["_id"]): {"title": row["title"], "text": row["text"]}
    for row in corpus_rows
}

# A capped corpus can drop the qrels' own relevant documents -- worth
# knowing up front rather than discovering silently degraded hit rates
# later. Not fatal (a smoke-test run doesn't need every relevant doc
# present), but flagged.
covered_relevant = sum(
    1 for docs in qrels.values() for doc_id in docs if doc_id in fiqa_corpus
)
print(f"Loaded {len(fiqa_corpus)} corpus documents.")
print(
    f"{covered_relevant}/{relevant_doc_count} relevant documents are present "
    "in this corpus slice."
)

## 5. Load an embedding model

Loads QAChat's embedding model, picking a backend (MLX or PyTorch) automatically -- needed early because `chunk_source_units` uses its real token counter for chunking.

In [ ]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
EMBED_MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}

In [ ]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
EMBED_MODEL_ID = EMBED_MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Embedding model: {EMBED_MODEL_ID}")

In [ ]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    EMBED_MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Embedding model loaded on: {embedding_session.device}")

## 6. Turn the corpus into a real QAChat project

`sandbox/fiqa_benchmark/` is a real QAChat project -- `profile.ini` plus a `documents/` source folder, `kind = literature` (QAChat's non-code project type; its default source-folder name is `documents`, not `repository`). Each corpus document is written as `documents/<doc_id>.txt` -- `title` prepended when FiQA actually has one (it's usually empty in this dataset) -- so `chunk.path.stem` recovers the original BEIR document id directly once real chunking runs, no separate id-mapping table needed for the common case.

`kind = literature` only affects `open_project`'s own bookkeeping and default folder naming here -- the actual retrieval query instruction is set explicitly, not left to QAChat's kind-driven default.


In [ ]:
from benchmarks.common import corpus_already_written
from qachat.project import configure_project, initialize_project

project_dir = root / "benchmarks" / "sandbox" / "fiqa_benchmark"
documents_dir = project_dir / "documents"

if corpus_already_written(documents_dir):
    print(f"Corpus already present under {documents_dir}, skipping write.")
else:
    documents_dir.mkdir(parents=True, exist_ok=True)
    for doc_id, doc in fiqa_corpus.items():
        text = (
            f"{doc['title']}\n\n{doc['text']}" if doc["title"] else doc["text"]
        )
        (documents_dir / f"{doc_id}.txt").write_text(text, encoding="utf-8")
    print(
        f"Wrote {len(fiqa_corpus)} documents "
        f"to {documents_dir.relative_to(root)}"
    )

# initialize_project writes a default profile.ini only if one is missing, then
# opens the project. The default extensions (".pdf, .md, .txt") and
# max_file_bytes (20 MB, generous for literature) are broader than this .txt
# corpus needs, so narrow them explicitly.
project = initialize_project(project_dir, "literature")
if (
    project.profile.extensions != (".txt",)
    or project.profile.max_file_bytes != 1_048_576
):
    project = configure_project(
        project_dir,
        {
            "extensions": ".txt",
            "max_file_bytes": 1_048_576,
        },
    )

## 7. Build the retrieval corpus with QAChat's real pipeline

`scan_repository` -> `read_repository_files` -> `extract_source_units` -> `chunk_source_units`, exactly as `RAGSession._prepare_source` does for any real project, run here over prose instead of source code. No code graph is built (see the intro) -- there's nothing for it to index.

`scan_repository` defaults to `max_entries=10_000` (an interactive-project safety cap) and raises `ValueError` once it does, rather than truncating silently -- well below FiQA's 57,638 documents once `CORPUS_SAMPLE_SIZE`
is `None`, so we size `max_entries` to `len(fiqa_corpus) + 100` below (the scan is flat, one entry per document) so it scales with `CORPUS_SAMPLE_SIZE` and a full-corpus run actually scans every document instead of stopping partway.


In [ ]:
from qachat.project import open_project

project = open_project(project_dir)
profile = project.profile

print(f"Project source: {project.source}")
print(
    f"Chunking: target={profile.target_tokens} max={profile.max_tokens} "
    f"overlap={profile.overlap_tokens}"
)

In [ ]:
from qachat.repository import scan_repository
from qachat.sources import extract_source_units, read_repository_files
from qachat.chunking import chunk_source_units

# scan_repository defaults to max_entries=10_000 and raises ValueError past
# it (not a silent truncation) -- well below FiQA's 57,638 documents at
# CORPUS_SAMPLE_SIZE=None. documents/ is flat (one .txt per corpus doc, no
# subfolders), so the scan visits exactly len(fiqa_corpus) entries -- size
# the cap to that, with headroom, so it scales with CORPUS_SAMPLE_SIZE
# instead of a fixed number.
repository_report = scan_repository(
    project.source,
    extensions=profile.extensions,
    max_file_bytes=profile.max_file_bytes,
    max_entries=len(fiqa_corpus) + 100,
)
source_files = read_repository_files(repository_report)
source_units = extract_source_units(source_files)
fiqa_chunks = chunk_source_units(
    source_units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=profile.target_tokens,
    max_tokens=profile.max_tokens,
    overlap_tokens=profile.overlap_tokens,
)

multi_chunk_docs = len(fiqa_corpus) - len(
    {chunk.path.stem for chunk in fiqa_chunks}
)
print(f"Scanned {len(source_files)} files, {len(source_units)} source units.")
print(
    f"Built {len(fiqa_chunks)} chunks "
    f"({multi_chunk_docs} documents split into more than one chunk)."
)

## 8. Build a whole-document baseline (no splitting)

Build one unsplit chunk per corpus document -- the same granularity BEIR's own published table uses. FiQA's posts are short (median ~500 characters here, well under `target_tokens = 300`'s token budget) with no long-tail outlier problem like ODEX's or DS-1000's, so no truncation cap is needed. A handful of documents (3 out of 2,000 in a smoke-test sample) have empty title and text -- real chunking already drops these silently (`qachat/chunking.py` filters blank text at several points), so this baseline skips them too, rather than trying to embed an empty string. Chunks are still sorted by length before embedding (section 11), so batching doesn't pad the many short posts up to the rare longer one's size.


In [ ]:
from qachat.chunking import Chunk


def build_whole_document_chunks() -> list[Chunk]:
    """Build one whole, unsplit chunk per corpus document.

    Real chunking (section 7) splits long posts into paragraph-sized
    pieces on its own, and silently drops any unit whose text is blank
    after stripping (qachat/chunking.py filters this at both the source-
    unit and final-chunk level). This baseline can't split anything --
    it's one chunk per document by definition -- but it still has to
    apply that same empty-text filter itself, or a handful of genuinely
    blank FiQA documents (no title, no text) would reach the embedding
    model as empty strings and raise ValueError.
    """

    chunks = []
    for doc_id, doc in fiqa_corpus.items():
        text = (
            f"{doc['title']}\n\n{doc['text']}" if doc["title"] else doc["text"]
        )
        if not text.strip():
            continue
        chunks.append(
            Chunk(
                identifier=doc_id,
                path=Path(f"{doc_id}.txt"),
                text=text,
                start_line=1,
                end_line=text.count("\n") + 1,
                kind="whole_document",
                symbol=doc["title"] or None,
                section=None,
            )
        )
    # Sorted by length, not corpus order: batching (section 11) pads every
    # sequence in a batch to its longest member, so leaving a handful of
    # longer posts next to dozens of short ones wastes some of that
    # batch's compute on padding. Grouping similar lengths together keeps
    # batches close to their true cost instead.
    chunks.sort(key=lambda chunk: len(chunk.text))
    return chunks


whole_document_chunks = build_whole_document_chunks()
skipped_empty = len(fiqa_corpus) - len(whole_document_chunks)
print(
    f"fiqa: {len(whole_document_chunks)} whole-document chunks "
    f"({skipped_empty} documents skipped: empty title and text)"
)

## 9. Run retrieval

Define semantic, lexical, and hybrid retrieval as reusable functions, then build the real-chunking index once. No code channel: FiQA's corpus is financial forum prose, not parseable code, so there's nothing for `code_search` to resolve.

FiQA's qrels and the published NDCG@10 table are both document-level, while QAChat's retrieval identity is `Chunk.identifier` -- so each function's raw chunk-level results are max-aggregated to one best-scoring chunk per document id (`chunk.path.stem`) before being returned, the same logic the notebook used inline before this section existed.

Embedding is the slow part -- so the index is saved under the project's own `indexes/` folder (the same mechanism `RAGSession` itself uses) and reloaded next time if its fingerprint still matches, rebuilt otherwise.


In [ ]:
import time

from tqdm.auto import tqdm

from qachat import (
    build_lexical_index,
    build_semantic_index,
    hybrid_search,
    indexes_are_current,
    lexical_search,
    load_indexes,
    prepare_persistent_indexes,
    save_indexes,
    semantic_matrix,
    semantic_search,
)
from qachat.query_formats import format_qwen3_query
from qachat.retrievals.code_graph.code_graph import CodeGraph

EMBED_BATCH_SIZE = 32
CANDIDATE_COUNT = 100
QUERY_SAMPLE_SIZE = None  # None for all test queries

sampled_query_ids = (
    list(fiqa_queries)
    if QUERY_SAMPLE_SIZE is None
    else list(fiqa_queries)[:QUERY_SAMPLE_SIZE]
)

FIQA_RETRIEVAL_INSTRUCTION = (
    "Given a financial question, retrieve relevant forum passages that "
    "answer the question."
)

# No code entities exist for FiQA's prose corpus, so the
# fingerprint/caching machinery below -- built for QAChat's real
# semantic+lexical+code indexes -- is given a permanently empty code graph.
empty_code_graph = CodeGraph(entities=(), relations=())


def _max_aggregate_by_document(ranked) -> dict[str, object]:
    """Collapse chunk-level search results to best-scoring res per document id.

    Otherwise one document could occupy two slots in a query's top-k and
    silently crowd out a different, actually-distinct relevant document.
    """

    doc_best: dict[str, object] = {}
    for result in ranked:
        doc_id = result.chunk.path.stem
        if doc_id not in doc_best or result.score > doc_best[doc_id].score:
            doc_best[doc_id] = result
    return doc_best


def run_semantic_retrieval(
    semantic_index, *, desc: str, matrix=None, count: int = CANDIDATE_COUNT
) -> dict[str, dict[str, float]]:
    """Rank every sampled query's documents with QAChat's semantic search."""

    search_kwargs = (
        {"backend": "numpy", "matrix": matrix} if matrix is not None else {}
    )
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(
        sampled_query_ids, desc=f"{desc}: semantic", unit="query", leave=False
    )
    for query_id in query_items:
        formatted_query = format_qwen3_query(
            fiqa_queries[query_id], FIQA_RETRIEVAL_INSTRUCTION
        )
        ranked = semantic_search(
            semantic_index,
            formatted_query,
            embedding_session,
            count=count,
            **search_kwargs,
        )
        results[query_id] = {
            doc_id: r.score
            for doc_id, r in _max_aggregate_by_document(ranked).items()
        }
    return results


def run_lexical_retrieval(
    lexical_index, *, desc: str, count: int = CANDIDATE_COUNT
) -> dict[str, dict[str, float]]:
    """Rank every sampled query's documents with QAChat's lexical search."""

    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(
        sampled_query_ids, desc=f"{desc}: lexical", unit="query", leave=False
    )
    for query_id in query_items:
        ranked = lexical_search(
            lexical_index, fiqa_queries[query_id], count=count
        )
        results[query_id] = {
            doc_id: r.score
            for doc_id, r in _max_aggregate_by_document(ranked).items()
        }
    return results


def run_hybrid_retrieval(
    semantic_index,
    lexical_index,
    *,
    desc: str,
    matrix=None,
    count: int = CANDIDATE_COUNT,
    keep_context: bool = False,
):
    """Fuse semantic and lexical rankings, max-aggregated per document id."""

    search_kwargs = (
        {"backend": "numpy", "matrix": matrix} if matrix is not None else {}
    )
    results: dict[str, dict[str, float]] = {}
    context: dict[str, list[dict[str, object]]] = {}
    query_items = tqdm(
        sampled_query_ids, desc=f"{desc}: hybrid", unit="query", leave=False
    )
    for query_id in query_items:
        query_text = fiqa_queries[query_id]
        formatted_query = format_qwen3_query(
            query_text, FIQA_RETRIEVAL_INSTRUCTION
        )
        semantic_ranked = semantic_search(
            semantic_index,
            formatted_query,
            embedding_session,
            count=count,
            **search_kwargs,
        )
        lexical_ranked = lexical_search(lexical_index, query_text, count=count)
        combined = hybrid_search(
            semantic=semantic_ranked, lexical=lexical_ranked, count=count
        )
        doc_best = _max_aggregate_by_document(combined)
        results[query_id] = {doc_id: r.score for doc_id, r in doc_best.items()}
        if keep_context:
            context[query_id] = [
                {
                    "doc_id": doc_id,
                    "score": r.score,
                    "channels": list(r.channels),
                    "is_relevant": qrels.get(query_id, {}).get(doc_id, 0) > 0,
                }
                for doc_id, r in sorted(
                    doc_best.items(),
                    key=lambda item: item[1].score,
                    reverse=True,
                )
            ]
    return (results, context) if keep_context else results


index_path = project.indexes / "fiqa_retrieval.json"
cached = None
if index_path.exists():
    try:
        cached = load_indexes(index_path)
    except ValueError:
        cached = None
    if cached is not None and not indexes_are_current(
        cached, fiqa_chunks, empty_code_graph
    ):
        cached = None

if cached is not None:
    corpus_semantic_index = cached.semantic
    corpus_lexical_index = cached.lexical
    print(
        f"fiqa: loaded cached index ({len(fiqa_chunks)} chunks) "
        f"from {index_path.relative_to(root)}"
    )
else:
    with tqdm(
        total=len(fiqa_chunks), desc="Embedding fiqa chunks"
    ) as progress_bar:

        def _on_embed_progress(current: int, total: int) -> None:
            progress_bar.n = current
            progress_bar.refresh()

        corpus_semantic_index = build_semantic_index(
            fiqa_chunks,
            embedding_session,
            batch_size=EMBED_BATCH_SIZE,
            progress_callback=_on_embed_progress,
        )

    _lexical_start = time.monotonic()
    corpus_lexical_index = build_lexical_index(fiqa_chunks)
    print(
        "fiqa: lexical index built in "
        f"{time.monotonic() - _lexical_start:.1f}s."
    )

    save_indexes(
        prepare_persistent_indexes(
            corpus_semantic_index, corpus_lexical_index, empty_code_graph
        ),
        index_path,
    )
    print(
        f"fiqa: indexed {len(fiqa_chunks)} chunks, "
        f"saved to {index_path.relative_to(root)}"
    )

corpus_matrix = semantic_matrix(corpus_semantic_index)

## 10. Score against the published range

Scores against document-level qrels via `evaluate_retrieval` -- BEIR's own primary metric is NDCG@10. Semantic and lexical are scored alongside hybrid; with only two real channels, this table is already the full picture.


In [ ]:
from benchmarks.common import evaluate_retrieval

sampled_qrels = {qid: qrels[qid] for qid in sampled_query_ids}

semantic_results = run_semantic_retrieval(
    corpus_semantic_index, desc="fiqa", matrix=corpus_matrix
)
lexical_results = run_lexical_retrieval(corpus_lexical_index, desc="fiqa")
hybrid_results, retrieved_context_by_query = run_hybrid_retrieval(
    corpus_semantic_index,
    corpus_lexical_index,
    desc="fiqa",
    matrix=corpus_matrix,
    keep_context=True,
)

fiqa_scores = {
    "semantic": evaluate_retrieval(sampled_qrels, semantic_results),
    "lexical": evaluate_retrieval(sampled_qrels, lexical_results),
    "hybrid": evaluate_retrieval(sampled_qrels, hybrid_results),
}

for channel, channel_scores in fiqa_scores.items():
    print(
        f"{channel:>10}: NDCG@10={channel_scores['ndcg']['NDCG@10']:.3f}  "
        f"MRR@10={channel_scores['mrr']['MRR@10']:.3f}  "
        f"Recall@10={channel_scores['recall']['Recall@10']:.3f}"
    )

## 11. Compare: whole-document baseline vs. QAChat chunking

`whole_document` matches BEIR's own published granularity exactly -- one embedding per document, the same unit the table below reports. `qachat_chunking` is `fiqa_scores["hybrid"]`, QAChat's real pipeline. Any gap between the two is a direct consequence of splitting some documents into more than one chunk.


In [ ]:
with tqdm(
    total=len(whole_document_chunks), desc="Embedding fiqa whole documents"
) as progress_bar:

    def _on_embed_progress(current: int, total: int) -> None:
        progress_bar.n = current
        progress_bar.refresh()

    whole_document_semantic_index = build_semantic_index(
        whole_document_chunks,
        embedding_session,
        batch_size=EMBED_BATCH_SIZE,
        progress_callback=_on_embed_progress,
    )
_lexical_start = time.monotonic()
whole_document_lexical_index = build_lexical_index(whole_document_chunks)
print(
    "fiqa whole-document: lexical index built in "
    f"{time.monotonic() - _lexical_start:.1f}s."
)
whole_document_matrix = semantic_matrix(whole_document_semantic_index)

whole_document_results = run_hybrid_retrieval(
    whole_document_semantic_index,
    whole_document_lexical_index,
    desc="fiqa whole-document",
    matrix=whole_document_matrix,
)

comparison_scores = {
    "whole_document": evaluate_retrieval(
        sampled_qrels, whole_document_results
    ),
    "qachat_chunking": fiqa_scores["hybrid"],
}

HEADLINE_MEASURES = (
    ("mrr", "MRR@10"),
    ("ndcg", "NDCG@10"),
    ("recall", "Recall@10"),
    ("recall", "Recall@100"),
)

header = ["mode", *(label for _, label in HEADLINE_MEASURES)]
print(" | ".join(f"{h:>14}" for h in header))
for mode in ("whole_document", "qachat_chunking"):
    mode_scores = comparison_scores[mode]
    values = [
        f"{mode_scores[group][label]:.3f}"
        for group, label in HEADLINE_MEASURES
    ]
    print(" | ".join(f"{cell:>14}" for cell in [mode, *values]))

**Reading the two numbers.** Unlike ODEX or DS-1000, `qachat_chunking` barely trails `whole_document` here. That tracks with the previous finding: only about one document in nine gets split into more than one chunk, so most queries see the same retrieval unit either way -- there's little room for chunking to cost anything on a corpus this short and uniform.

Both numbers land in the same place against BEIR's own published range below: ahead of ColBERT and every dense retriever they benchmarked, behind only the cross-encoder reranker BM25+CE.

BEIR's own published NDCG@10 for FiQA-2018 (Thakur et al., 2021, Table 2), across every retriever they benchmarked:

| Retriever | NDCG@10 |
|---|---|
| DPR | 0.112 |
| BM25 | 0.236 |
| ANCE | 0.295 |
| TAS-B | 0.300 |
| GenQ | 0.308 |
| ColBERT | 0.317 |
| BM25+CE | 0.347 |


## 12. Save results


In [ ]:
import json as _json
from benchmarks.common import local_run_timestamp

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "BEIR (retrieval only) -- FiQA-2018, QAChat's real pipeline",
    "timestamp": timestamp,
    "embedding_model": EMBED_MODEL_ID,
    "backend": selected_backend,
    "corpus_sample_size": CORPUS_SAMPLE_SIZE,
    "query_sample_size": QUERY_SAMPLE_SIZE,
    "candidate_count": CANDIDATE_COUNT,
    "retrieval_channels": ["semantic", "lexical"],
    "retrieval_instruction": FIQA_RETRIEVAL_INSTRUCTION,
    "chunking_profile": {
        "target_tokens": profile.target_tokens,
        "max_tokens": profile.max_tokens,
        "overlap_tokens": profile.overlap_tokens,
    },
    "corpus_document_count": len(fiqa_corpus),
    "corpus_chunk_count": len(fiqa_chunks),
    "query_count": len(sampled_query_ids),
    "scores": fiqa_scores,
    "comparison_scores": comparison_scores,
    "per_query_retrieved": retrieved_context_by_query,
}

output_path = results_dir / f"{stamp}_beir_fiqa_retrieval.json"
output_path.write_text(_json.dumps(run_record, indent=2))

print(f"Saved: {output_path.relative_to(root)}")

## 13. Release the models


In [ ]:
embedding_session.close()